# Seguimiento y caracterización de satélites
## Etapa 1 — Análisis instantáneo del catálogo


In [ ]:
from datetime import datetime, timezone

from funciones_base import epoch_medio
from catalogo import leer_satelites, contar_por_grupo, lanzamientos_por_anio
from satelite import crear_satelites
from analisis import (
    contar_por_tipo,
    mas_bajo,
    mas_alto,
    top_n,
    resumen_por_grupo,
    en_zona,
    ordenar_por
)
from catalogo import leer_satelites, contar_por_grupo, lanzamientos_por_anio, grupo
from graficos import (
    graficar_lanzamientos,
    graficar_lanzamientos_apilados,
    graficar_altitud_media_por_grupo,
    graficar_histograma_leo,
)

# Instante de cálculo actual en UTC
momento_actual = datetime.now(timezone.utc)
print(f"Momento de cálculo: {momento_actual.isoformat(timespec='seconds')}")

Momento de cálculo: 2026-10-29T01:11:55+00:00


### 1. Carga de datos

Leemos el catálogo oficial en formato JSON y evaluamos la antigüedad media de las observaciones astronómicas registradas (`EPOCH`). Además, filtramos aquellos objetos cuya altitud dé por debajo de los 100 km (objetos que han reingresado a la atmósfera o cuyos elementos orbitales carecen de precisión física actual).

In [2]:
registros = leer_satelites("datos/satelites.json")
total_registros = len(registros)

# Fecha media del catálogo y cálculo de días transcurridos
fecha_epoch = epoch_medio(registros)
dias_pasados = (momento_actual - fecha_epoch).total_seconds() / 86400

# Creación de objetos satelitales y descarte de reingresos (< 100 km)
satelites_activos = crear_satelites(registros, momento_actual)
total_activos = len(satelites_activos)
descartados = total_registros - total_activos

print(f"Total de satélites en el archivo: {total_registros}")
print(f"Fecha media del catálogo (EPOCH medio): {fecha_epoch.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Días transcurridos desde el EPOCH medio: {dias_pasados:.1f} días")
print(f"Satélites válidos activos calculados: {total_activos}")
print(f"Satélites descartados (< 100 km de altitud): {descartados}")

Total de satélites en el archivo: 16822
Fecha media del catálogo (EPOCH medio): 2026-10-27 00:56:47 UTC
Días transcurridos desde el EPOCH medio: 2.0 días
Satélites válidos activos calculados: 16822
Satélites descartados (< 100 km de altitud): 0


### 2. Grupos de satélites

Analizamos la distribución de la población satelital según su constelación u operador principal, prestando especial atención a la proporción que representa la constelación Starlink respecto al total del catálogo.

In [3]:
conteo_grupos = contar_por_grupo(registros)

for grp, cant in sorted(conteo_grupos.items(), key=lambda x: x[1], reverse=True):
    porcentaje = (cant / total_registros) * 100
    print(f"- {grp:12s}: {cant:5d} ({porcentaje:5.2f}%)")

cant_starlink = conteo_grupos.get("Starlink", 0)
pct_starlink = (cant_starlink / total_registros) * 100
print(f"\nStarlink representa el {pct_starlink:.2f}% del catálogo total.")

- Starlink    : 11160 (66.34%)
- Otros       :  4152 (24.68%)
- OneWeb      :   651 ( 3.87%)
- Kuiper      :   391 ( 2.32%)
- Qianfan     :   257 ( 1.53%)
- Iridium     :    80 ( 0.48%)
- BeiDou      :    55 ( 0.33%)
- GPS         :    40 ( 0.24%)
- Globalstar  :    36 ( 0.21%)

Starlink representa el 66.34% del catálogo total.


### 3. Lanzamientos por año

Examinamos la cantidad de satélites que continúan en órbita según el año en que fueron puestos en lanzamiento, considerando los registros desde 2015 hasta el presente.

In [4]:
lanzamientos = lanzamientos_por_anio(registros)

print("Lanzamientos de satélites activos por ano (desde 2015):")
anos_desde_2015 = [a for a in sorted(lanzamientos.keys()) if a >= 2015]

for ano in anos_desde_2015:
    print(f"Ano {ano}: {lanzamientos[ano]} satélites")

# Identificar el ano récord dentro de este rango
ano_record = max(anos_desde_2015, key=lambda a: lanzamientos[a])
print(f"\nEl ano con mayor cantidad de satélites activos lanzados fue {ano_record} con {lanzamientos[ano_record]} unidades.")

Lanzamientos de satélites activos por ano (desde 2015):
Ano 2015: 96 satélites
Ano 2016: 84 satélites
Ano 2017: 136 satélites
Ano 2018: 193 satélites
Ano 2019: 144 satélites
Ano 2020: 354 satélites
Ano 2021: 987 satélites
Ano 2022: 1715 satélites
Ano 2023: 2260 satélites
Ano 2024: 2428 satélites
Ano 2025: 4289 satélites
Ano 2026: 3421 satélites

El ano con mayor cantidad de satélites activos lanzados fue 2025 con 4289 unidades.


### 4. Tipos de órbita

Clasificación de los satélites activos de acuerdo con su altitud sobre la superficie terrestre:
* **LEO** (Órbita baja): < 2.000 km
* **MEO** (Órbita media): 2.000 km a < 34.000 km
* **GEO** (Geoestacionaria): 34.000 km a 37.000 km
* **HEO** (Alta / otras): > 37.000 km

In [5]:
distribucion_orbitas = contar_por_tipo(satelites_activos)

for orbita, cantidad in distribucion_orbitas.items():
    pct = (cantidad / total_activos) * 100
    print(f"Órbita {orbita:4s}: {cantidad:5d} satélites ({pct:5.2f}%)")

Órbita LEO : 16014 satélites (95.20%)
Órbita MEO :   195 satélites ( 1.16%)
Órbita GEO :   591 satélites ( 3.51%)
Órbita HEO :    22 satélites ( 0.13%)


### 5. Extremos de altitud

Identificación de los satélites en los límites orbitales: el satélite a menor altitud, el de mayor altitud y los listados con las diez mayores y diez menores alturas registradas.

In [6]:
sat_min = mas_bajo(satelites_activos)
sat_max = mas_alto(satelites_activos)

print(f"Satélite más bajo: {sat_min}")
print(f"Satélite más alto: {sat_max}\n")

print("Top 10 satélites más bajos:")
for s in top_n(satelites_activos, 10, mas_altos=False):
    print(" ", s)

print("\nTop 10 satélites más altos:")
for s in top_n(satelites_activos, 10, mas_altos=True):
    print(" ", s)

Satélite más bajo: STARLINK-1036 | LEO | 132.8 km
Satélite más alto: MMS 4 | HEO | 130810.8 km

Top 10 satélites más bajos:
  STARLINK-1036 | LEO | 132.8 km
  IPERDRONE.0 | LEO | 134.2 km
  RASSVET-3 23 | LEO | 138.0 km
  RASSVET-3 17 | LEO | 186.0 km
  STARLINK-1309 | LEO | 195.5 km
  IOD-1 | LEO | 216.9 km
  STARLINK-1283 | LEO | 231.5 km
  STARLINK-3306 | LEO | 235.9 km
  STARLINK-1327 | LEO | 241.3 km
  STARLINK-1420 | LEO | 242.3 km

Top 10 satélites más altos:
  MMS 4 | HEO | 130810.8 km
  MMS 2 | HEO | 130796.1 km
  MMS 3 | HEO | 130794.1 km
  MMS 1 | HEO | 130744.4 km
  XMM-NEWTON | HEO | 91416.8 km
  CXO | HEO | 89575.9 km
  THEMIS A | HEO | 77539.5 km
  ASTRANIS UTILITYSAT | HEO | 66141.8 km
  PROBA-3 CSC | HEO | 59788.2 km
  PROBA-3 OSC | HEO | 59788.1 km


### 6. Comparación estadística por grupo

Resumen de altitud (mínima, media y máxima) discriminado por cada constelación u operador.

In [7]:
resumen = resumen_por_grupo(satelites_activos)

print(f"{'Grupo':12s} | {'Cantidad':8s} | {'Mínima (km)':12s} | {'Media (km)':12s} | {'Máxima (km)':12s}")
print("-" * 65)

for grp, metricas in sorted(resumen.items(), key=lambda x: x[1]['cantidad'], reverse=True):
    print(f"{grp:12s} | {metricas['cantidad']:8d} | {metricas['minima']:12.1f} | {metricas['media']:12.1f} | {metricas['maxima']:12.1f}")

Grupo        | Cantidad | Mínima (km)  | Media (km)   | Máxima (km) 
-----------------------------------------------------------------
Starlink     |    11160 |        132.8 |        468.1 |        597.9
Otros        |     4152 |        134.2 |       6262.0 |     130810.8
OneWeb       |      651 |        555.3 |       1210.0 |       1251.8
Kuiper       |      391 |        451.9 |        633.2 |        656.7
Qianfan      |      257 |        785.9 |       1043.6 |       1106.7
Iridium      |       80 |        632.5 |        777.2 |        807.9
BeiDou       |       55 |      21502.2 |      27497.2 |      36081.4
GPS          |       40 |      19710.7 |      20257.0 |      21044.3
Globalstar   |       36 |        691.9 |       1301.4 |       1832.9


### 7. Satélites sobre la zona de Bariloche

Filtramos los satélites que se encuentran dentro del cuadro delimitado para la región de San Carlos de Bariloche: `lon_min = -76.30`, `lat_min = -46.13`, `lon_max = -66.30`, `lat_max = -36.13`.

In [8]:
BARILOCHE = (-76.30, -46.13, -66.30, -36.13)

satelites_bariloche = en_zona(satelites_activos, BARILOCHE)
satelites_bariloche_ordenados = ordenar_por(satelites_bariloche, "altitud_km", descendente=False)

print(f"Cantidad de satélites sobrevolando Bariloche en este instante: {len(satelites_bariloche_ordenados)}\n")
print("Listado ordenado por altitud (menor a mayor):")
for s in satelites_bariloche_ordenados:
    print(" ", s)

Cantidad de satélites sobrevolando Bariloche en este instante: 39

Listado ordenado por altitud (menor a mayor):
  STARLINK-11452 [DTC] | LEO | 365.6 km
  STARLINK-11150 [DTC] | LEO | 372.0 km
  STARLINK-34104 | LEO | 475.3 km
  STARLINK-35904 | LEO | 475.3 km
  STARLINK-33743 | LEO | 476.4 km
  STARLINK-37358 | LEO | 477.2 km
  STARLINK-32007 | LEO | 477.6 km
  STARLINK-34505 | LEO | 478.3 km
  STARLINK-35496 | LEO | 478.4 km
  STARLINK-4620 | LEO | 480.7 km
  STARLINK-34749 | LEO | 481.3 km
  STARLINK-37393 | LEO | 481.7 km
  STARLINK-38098 | LEO | 482.7 km
  STARLINK-34909 | LEO | 483.1 km
  STARLINK-34963 | LEO | 483.7 km
  STARLINK-33991 | LEO | 483.7 km
  STARLINK-34950 | LEO | 483.8 km
  STARLINK-37011 | LEO | 484.0 km
  STARLINK-34023 | LEO | 486.5 km
  STARLINK-6099 | LEO | 488.1 km
  STARLINK-35426 | LEO | 488.2 km
  STARLINK-35931 | LEO | 491.8 km
  STARLINK-30847 | LEO | 491.9 km
  STARLINK-35437 | LEO | 492.1 km
  STARLINK-36463 | LEO | 493.2 km
  STARLINK-30077 | LEO | 49

### 8. Conclusiones

A partir del procesamiento instantáneo de los 16.822 registros del catálogo de CelesTrak, se observa un marcado predominio de la constelación **Starlink**, la cual concentra más del 65% de la totalidad de satélites activos en órbita. Esta concentración se refleja directamente en la distribución orbital, donde la categoría **LEO (órbita baja)** reúne a la abrumadora mayoría de los dispositivos, operando predominantemente entre los 500 y 600 km de altitud. La serie temporal de lanzamientos evidencia una aceleración exponencial a partir de los anos 2020 y 2021, alcanzando sus picos de despliegue en anos recientes producto de la puesta en marcha de megaconstelaciones comerciales de telecomunicaciones. Finalmente, la ventana instantánea analizada sobre el cuadrante geográfico de **Bariloche** corrobora la densidad del tráfico en órbita baja, registrando una presencia activa y continua de satélites sobre la región patagónica.

---
## Etapa 2 — Visualización y análisis estadístico

### 1. Lanzamientos por año (1990–2026)

Analiza la evolución histórica de los lanzamientos de satélites activos y comparamos el peso de la constelación Starlink frente al resto de los operadores.

In [2]:
# Gráfico 1A: Lanzamientos totales por ano (1990 - 2026)
fig1, ax1 = graficar_lanzamientos(lanzamientos, desde=1990, hasta=2026)

# Separamos registros de Starlink vs. resto para las barras apiladas
registros_starlink = [r for r in registros if grupo(r.get("OBJECT_NAME", "")) == "Starlink"]
registros_resto = [r for r in registros if grupo(r.get("OBJECT_NAME", "")) != "Starlink"]

lanz_starlink = lanzamientos_por_anio(registros_starlink)
lanz_resto = lanzamientos_por_anio(registros_resto)

# Gráfico 1B: Barras apiladas Starlink vs. Resto
fig2, ax2 = graficar_lanzamientos_apilados(
    lanz_starlink,
    lanz_resto,
    etiqueta1="Starlink",
    etiqueta2="Otros operadores",
    desde=1990,
    hasta=2026,
)

NameError: name 'graficar_lanzamientos' is not defined

**Análisis:** La escala de lanzamientos anuales cambia drásticamente a partir de **2020** (y se consolida desde 2021 en adelante), pasando de unos pocos cientos de satélites por ano a varios miles. Como muestran las barras apiladas, **Starlink explica la mayor parte de ese salto** (más del 65%–75% de los lanzamientos anuales recientes), aunque el resto de los operadores también acompañó con un crecimiento sostenido.

### 2. Altura media por grupo

Comparamos la altitud orbital promedio de cada constelación u operador identificado en el catálogo.

In [3]:
fig3, ax3 = graficar_altitud_media_por_grupo(resumen)

NameError: name 'graficar_altitud_media_por_grupo' is not defined

**Análisis:** Los grupos **Starlink, Qianfan, Kuiper, Iridium, OneWeb y Globalstar** operan íntegramente en **órbita baja (LEO)**, con medias ubicadas entre los 500 y los 1.400 km. Por el contrario, **GPS y BeiDou** se posicionan mucho más arriba, en **órbita media (MEO)** alrededor de los 20.000–22.000 km, mientras que la categoría **Otros** promedia valores intermedios al mezclar satélites LEO científicos con satélites geoestacionarios (GEO) a ~35.786 km.

### 3. Altitudes en órbita baja (LEO: 200 a 2.000 km)

Comparamos la distribución de altitudes en órbita baja utilizando dos resoluciones distintas de agrupamiento: intervalos de `10 km` y de `50 km`.

In [4]:
fig4, ax4 = graficar_histograma_leo(satelites_activos, ancho_bin=10)
fig5, ax5 = graficar_histograma_leo(satelites_activos, ancho_bin=50)

NameError: name 'graficar_histograma_leo' is not defined

**Análisis:** Con `ancho_bin=50` se observa una distribución más suavizada que resalta la gran masa general entre los 450 y 600 km, pero oculta la estructura interna; en cambio, con `ancho_bin=10` **se distinguen picos finos y separados** correspondientes a las distintas capas o *shells* orbitales de Starlink (alrededor de 480 km, 530 km y 550–560 km). La mayor concentración de satélites LEO se ubica claramente entre los **480 y 580 km**, con otro pico secundario notable cerca de los **1.200 km** (correspondiente a OneWeb).